In [1]:
!whoami
!hostname


kavipooj001
CCDS-TC1


In [2]:
ls


'main (1).ipynb'                   runjupyter.sh
 main.ipynb                       'SC4000 AMEX Project.ipynb'
 merged_and_aggregated_train.ftr   test_data.ftr
 output_RunJupy8891_40321.out      train_data.ftr
 output_RunJupyter_40318.out       train_labels.csv
 runjupyter_8891.sh                train_merged_data.ftr


Merge customer_ID with the target value

In [3]:
import pandas as pd

# Load your dataset
train = pd.read_feather("merged_and_aggregated_train.ftr")

# Select only the relevant columns
subset = train[['customer_ID', 'target']]

# Print them all
print(subset)


                                              customer_ID  target
0       0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...       0
1       00000fd6641609c6ece5454664794f0340ad84dddce9a2...       0
2       00001b22f846c82c51f6e3958ccd81970162bae8b007e8...       0
3       000041bdba6ecadd89a52d11886e8eaaec9325906c9723...       0
4       00007889e4fcd2614b6cbe7f8f3d2e5c728eca32d9eb8a...       0
...                                                   ...     ...
458908  ffff41c8a52833b56430603969b9ca48d208e7c192c6a4...       0
458909  ffff518bb2075e4816ee3fe9f3b152c57fc0e6f01bf7fd...       0
458910  ffff9984b999fccb2b6127635ed0736dda94e544e67e02...       0
458911  ffffa5c46bc8de74f5a4554e74e239c8dee6b9baf38814...       1
458912  fffff1d38b785cef84adeace64f8f83db3a0c31e8d92ea...       0

[458913 rows x 2 columns]


In [4]:
categorical_df = train.select_dtypes(include=['object', 'category']).columns
print(categorical_df.tolist())

['customer_ID', 'B_30_last', 'B_38_last', 'D_114_last', 'D_116_last', 'D_117_last', 'D_120_last', 'D_126_last', 'D_63_last', 'D_64_last', 'D_66_last', 'D_68_last']


fix the missing values 

In [5]:
import pandas as pd
import gc
from IPython.display import display

# -----------------------------
# Load dataset
# -----------------------------
train = pd.read_feather("merged_and_aggregated_train.ftr")

# Drop old 'index' column if exists
if 'index' in train.columns:
    train = train.drop(columns=['index'])

# -----------------------------
# Identify columns with NaNs
# -----------------------------
nan_columns_before = train.columns[train.isna().any()].tolist()
print(f"Columns with NaN before imputation ({len(nan_columns_before)} columns):")
display(pd.DataFrame(nan_columns_before, columns=['Column Name']))

# -----------------------------
# Categorical features (fill with mode)
# -----------------------------
cat_features = ['S_2', 'B_30_last', 'B_38_last', 'D_114_last', 'D_116_last', 'D_117_last', 'D_120_last', 'D_126_last', 'D_63_last', 'D_64_last', 'D_66_last', 'D_68_last']

# -----------------------------
# Impute NaNs
# -----------------------------
for i, col_name in enumerate(nan_columns_before, 1):
    col = train[col_name]
    original_nan_count = col.isna().sum()
    
    # Temporarily convert float16 to float32 for safer computation
    convert_dtype = False
    if col.dtype == 'float16':
        col = col.astype('float32')
        convert_dtype = True
    
    # Impute
    if col_name in cat_features or col.dtype.name == 'category':
        col = col.fillna(col.value_counts().idxmax())
    else:
        col = col.fillna(col.mean())
    
    # Assign back to DataFrame
    train[col_name] = col

# -----------------------------
# Confirm all NaNs are replaced
# -----------------------------
nan_columns_after = train.columns[train.isna().any()].tolist()
print(f"\nColumns with NaN after imputation: {len(nan_columns_after)}")
display(pd.DataFrame(nan_columns_after, columns=['Column Name']))

# -----------------------------
# Save back to Feather
# -----------------------------
train.to_feather("train_merged_data.ftr")
print("\n✅ All NaNs replaced. Feather file overwritten.")

# Optional: free memory
gc.collect()


Columns with NaN before imputation (586 columns):


,Column Name
0,P_2_mean
1,P_2_std
2,P_2_min
3,P_2_max
4,P_2_last
...,...
581,D_117_last
582,D_120_last
583,D_64_last
584,D_66_last



Columns with NaN after imputation: 0


,Column Name



✅ All NaNs replaced. Feather file overwritten.


0

one hot encoding

In [6]:
import pandas as pd
import gc
from sklearn.preprocessing import OneHotEncoder
from IPython.display import display

# -----------------------------
# Function to get top influential categorical columns
# -----------------------------
def get_influential_cols(df, cat_cols, target='target', top_n=3):
    influence_dict = {}
    for col in cat_cols:
        means = df.groupby(col)[target].mean()
        influence_range = means.max() - means.min()
        influence_dict[col] = influence_range
    # Sort columns by influence
    top_cols = sorted(influence_dict, key=influence_dict.get, reverse=True)[:top_n]
    
    print(f"Top {top_n} columns selected for encoding: {top_cols}")
    
    # Display influence ranges nicely in a table
    influence_df = pd.DataFrame.from_dict(influence_dict, orient='index', columns=['Influence Range'])
    display(influence_df.sort_values('Influence Range', ascending=False).head(top_n))
    
    return top_cols

# -----------------------------
# One-hot encoding function
# -----------------------------
def one_hot_encode(df, cols_to_encode):
    enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    encoded_array = enc.fit_transform(df[cols_to_encode])
    encoded_cols = enc.get_feature_names_out(cols_to_encode)
    encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=df.index)
    
    print(f"One-hot encoding done for columns: {cols_to_encode}")
    return encoded_df

# -----------------------------
# Safe imputation function
# -----------------------------
def impute_columns(df, cat_cols):
    nan_summary = []
    for col in df.columns:
        nan_count = df[col].isna().sum()
        if col in cat_cols or df[col].dtype.name == 'category' or df[col].dtype == object:
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].astype('float64')
            df[col] = df[col].fillna(df[col].mean())
        nan_summary.append((col, nan_count))
    
    # Display a summary table of imputed columns
    summary_df = pd.DataFrame(nan_summary, columns=['Column', 'NaNs Filled'])
    display(summary_df[summary_df['NaNs Filled'] > 0])
    
    return df

# -----------------------------
# Full preprocessing workflow
# -----------------------------
def generate_x_y(df_file_path, cat_cols, target_col='target', top_n=3, test=False):
    df = pd.read_feather(df_file_path)
    
    # Impute all columns safely
    df = impute_columns(df, cat_cols)
    
    # Overwrite the original file with cleaned data
    df.to_feather(df_file_path)
    
    # Separate target if training
    y = None if test else df[target_col]
    
    # Select top influential categorical columns
    top_cols = get_influential_cols(df, cat_cols, target=target_col, top_n=top_n)
    
    # One-hot encode top columns
    encoded_df = one_hot_encode(df, top_cols)
    
    # Drop top columns and target from original df
    drop_cols = top_cols + ([target_col] if target_col in df.columns else [])
    X = df.drop(drop_cols, axis=1)
    
    # Combine with one-hot encoded columns
    X = pd.concat([X, encoded_df], axis=1)
    
    del df, encoded_df
    gc.collect()
    
    return (X, y) if not test else X

# -----------------------------
# Usage in Notebook
# -----------------------------
cat_cols = ['S_2', 'B_30_last', 'B_38_last', 'D_114_last', 'D_116_last', 'D_117_last', 'D_120_last', 'D_126_last', 'D_63_last', 'D_64_last', 'D_66_last', 'D_68_last']

# Run preprocessing
X_train, y_train = generate_x_y("train_merged_data.ftr", cat_cols, target_col='target', top_n=3)

# Display a sample of processed features
display(X_train.head())
print("Aggregated numeric data saved to train_merged_data.ftr")


,Column,NaNs Filled


Top 3 columns selected for encoding: ['B_38_last', 'B_30_last', 'D_116_last']


/tmp/ipykernel_384989/2251508089.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  means = df.groupby(col)[target].mean()


,Influence Range
B_38_last,0.668833
B_30_last,0.482680
D_116_last,0.449579


One-hot encoding done for columns: ['B_38_last', 'B_30_last', 'D_116_last']


,customer_ID,S_2,P_2_mean,P_2_std,P_2_min,P_2_max,P_2_last,D_39_mean,D_39_std,D_39_min,...,B_38_last_3.0,B_38_last_4.0,B_38_last_5.0,B_38_last_6.0,B_38_last_7.0,B_30_last_0.0,B_30_last_1.0,B_30_last_2.0,D_116_last_0.0,D_116_last_1.0
0,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2018-03-13,0.933782,0.024194,0.868652,0.960449,0.934570,0.010702,0.024440,0.001082,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,00000fd6641609c6ece5454664794f0340ad84dddce9a2...,2018-03-25,0.899827,0.022097,0.861328,0.929199,0.880371,0.215171,0.199123,0.002224,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,2018-03-12,0.878418,0.028837,0.797852,0.904297,0.880859,0.004181,0.002759,0.000802,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,000041bdba6ecadd89a52d11886e8eaaec9325906c9723...,2018-03-29,0.598933,0.020082,0.567383,0.623535,0.621582,0.048873,0.088490,0.000660,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,00007889e4fcd2614b6cbe7f8f3d2e5c728eca32d9eb8a...,2018-03-30,0.891752,0.042316,0.805176,0.940430,0.872070,0.004644,0.002883,0.000030,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


Aggregated numeric data saved to train_merged_data.ftr


# CatBoost Model

In [7]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

In [8]:
def amex_metric(y_true, y_pred):

    """"Combines the top 4% default capture rate and normalized Gini coefficient."""

    y_true, y_pred = np.array(y_true), np.array(y_pred)   # Convert to numpy arrays

    order = np.argsort(-y_pred)
    y_true, y_pred = y_true[order], y_pred[order]

    weights = np.where(y_true == 0, 20, 1)
    cum_weights = np.cumsum(weights)
    cutoff = 0.04 * np.sum(weights)

    top_four = y_true[cum_weights <= cutoff]
    d_rate = np.sum(top_four) / np.sum(y_true)

    def weighted_gini(actual, pred, weight):
        order = np.argsort(pred)
        actual, weight = actual[order], weight[order]
        cum_weight = np.cumsum(weight)
        cum_actual = np.cumsum(actual * weight)
        gini_sum = np.sum((cum_actual / cum_actual[-1]) - (cum_weight / cum_weight[-1]))
        return gini_sum / len(actual)

    def normalized_weighted_gini(actual, pred, weight):
        return weighted_gini(actual, pred, weight) / weighted_gini(actual, actual, weight)

    gini = normalized_weighted_gini(y_true, y_pred, weights)

    amex_score = 0.5 * (d_rate + gini)
    return amex_score

In [9]:
#load data
df = pd.read_feather("train_merged_data.ftr")
X = df.drop(columns=['target', 'customer_ID'])
y = df['target']

In [10]:
#categorical columns evaluation
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype(str).fillna("missing")

print(f"Categorical columns ({len(cat_cols)}):", cat_cols)

Categorical columns (11): ['B_30_last', 'B_38_last', 'D_114_last', 'D_116_last', 'D_117_last', 'D_120_last', 'D_126_last', 'D_63_last', 'D_64_last', 'D_66_last', 'D_68_last']


In [11]:
# splitting of data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
# catBoost Model
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function='Logloss',
    eval_metric='AUC',
    cat_features=cat_cols,
    verbose=100
)

cat_model.fit(X_train, y_train, eval_set=(X_test, y_test))

0:	test: 0.9329930	best: 0.9329930 (0)	total: 334ms	remaining: 2m 46s
100:	test: 0.9570006	best: 0.9570006 (100)	total: 23.3s	remaining: 1m 31s
200:	test: 0.9587588	best: 0.9587588 (200)	total: 45.8s	remaining: 1m 8s
300:	test: 0.9594564	best: 0.9594564 (300)	total: 1m 8s	remaining: 45s
400:	test: 0.9598494	best: 0.9598494 (400)	total: 1m 30s	remaining: 22.3s
499:	test: 0.9601288	best: 0.9601288 (499)	total: 1m 52s	remaining: 0us

bestTest = 0.960128833
bestIteration = 499



In [13]:
#predictions
y_pred = cat_model.predict_proba(X_test)[:, 1]
amex_score = amex_metric(y_test.values, y_pred)
print(f"CatBoost AMEX Metric: {amex_score:.5f}")

CatBoost AMEX Metric: 0.79059


# LightGBM Model


In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

In [15]:
def amex_metric(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    order = np.argsort(-y_pred)
    y_true, y_pred = y_true[order], y_pred[order]

    weights = np.where(y_true == 0, 20, 1)
    cum_weights = np.cumsum(weights)
    cutoff = 0.04 * np.sum(weights)

    top_four = y_true[cum_weights <= cutoff]
    d_rate = np.sum(top_four) / np.sum(y_true)

    def weighted_gini(actual, pred, weight):
        order = np.argsort(pred)
        actual, weight = actual[order], weight[order]
        cum_weight = np.cumsum(weight)
        cum_actual = np.cumsum(actual * weight)
        gini_sum = np.sum((cum_actual / cum_actual[-1]) - (cum_weight / cum_weight[-1]))
        return gini_sum / len(actual)

    def normalized_weighted_gini(actual, pred, weight):
        return weighted_gini(actual, pred, weight) / weighted_gini(actual, actual, weight)

    gini = normalized_weighted_gini(y_true, y_pred, weights)
    return 0.5 * (d_rate + gini)

In [16]:
#load data 
df = pd.read_feather("train_merged_data.ftr")

In [17]:
#prepare data
target = "target"
id_col = "customer_ID"
ids = df[id_col].copy() if id_col in df.columns else None

X = df.drop(columns=[target, id_col] if id_col in df.columns else [target])
y = df[target].astype(int)

In [18]:
#handle date time columns
datetime_cols = X.select_dtypes(include=["datetime64[ns]", "datetime64"]).columns.tolist()
if datetime_cols:
    for col in datetime_cols:
        X[col] = pd.to_datetime(X[col], errors="coerce").astype("int64") // 10**9
    print(f"Converted datetime columns to numeric: {datetime_cols}")

Converted datetime columns to numeric: ['S_2']


In [20]:
# handle categorical columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype(str).fillna("missing")
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

In [21]:
# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [22]:
#lightGBM Model
lgbm = LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=-1,
    num_leaves=255,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    objective="binary",
    n_jobs=-1
)

lgbm.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="auc",
    callbacks=[early_stopping(stopping_rounds=300), log_evaluation(100)]
)

[LightGBM] [Info] Number of positive: 95062, number of negative: 272068
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.527129 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 220008
[LightGBM] [Info] Number of data points in the train set: 367130, number of used features: 914
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.258933 -> initscore=-1.051523
[LightGBM] [Info] Start training from score -1.051523
Training until validation scores don't improve for 300 rounds
[100]	valid_0's auc: 0.958842	valid_0's binary_logloss: 0.233796
[200]	valid_0's auc: 0.961154	valid_0's binary_logloss: 0.219724
[300]	valid_0's auc: 0.961903	valid_0's binary_logloss: 0.217181
[400]	valid_0's auc: 0.962111	valid_0's binary_logloss: 0.216573
[500]	valid_0's auc: 0.962157	valid_0's binary_logloss: 0.216519
[600]	valid_0's auc: 0.962207	valid_0's binary_logloss: 0.216572
[700]	valid_0's auc: 0.962256	valid_0's binary_lo

,boosting_type,'gbdt'
,num_leaves,255
,max_depth,-1
,learning_rate,0.03
,n_estimators,5000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [24]:
#predicitons
y_pred = lgbm.predict_proba(X_test)[:, 1]
print(f"LightGBM AMEX Metric: {amex_metric(y_test.values, y_pred):.5f}")

LightGBM AMEX Metric: 0.79416


# XGBoost Model

In [37]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [26]:
def amex_metric(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    order = np.argsort(-y_pred)
    y_true, y_pred = y_true[order], y_pred[order]

    weights = np.where(y_true == 0, 20, 1)
    cum_weights = np.cumsum(weights)
    cutoff = 0.04 * np.sum(weights)

    top_four = y_true[cum_weights <= cutoff]
    d_rate = np.sum(top_four) / np.sum(y_true)

    def weighted_gini(actual, pred, weight):
        order = np.argsort(pred)
        actual, weight = actual[order], weight[order]
        cum_weight = np.cumsum(weight)
        cum_actual = np.cumsum(actual * weight)
        gini_sum = np.sum((cum_actual / cum_actual[-1]) - (cum_weight / cum_weight[-1]))
        return gini_sum / len(actual)

    def normalized_weighted_gini(actual, pred, weight):
        return weighted_gini(actual, pred, weight) / weighted_gini(actual, actual, weight)

    gini = normalized_weighted_gini(y_true, y_pred, weights)
    return 0.5 * (d_rate + gini)

In [27]:
#load data
df = pd.read_feather("train_merged_data.ftr")

In [28]:
#prepare data
X = df.drop(columns=['target', 'customer_ID'])
y = df['target']

In [29]:
#handle data-time columns
datetime_cols = X.select_dtypes(include=['datetime64[ns]', 'datetime64']).columns.tolist()
if len(datetime_cols) > 0:
    for col in datetime_cols:
        X[col] = X[col].astype('int64') // 10**9  # Convert to UNIX timestamp
    print(f"Converted datetime columns to numeric: {datetime_cols}")


Converted datetime columns to numeric: ['S_2']


In [32]:
# handle categorical columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype(str).fillna("missing")
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

In [33]:
#split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [35]:
#XGBoost Model
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    tree_method='hist',  # use 'gpu_hist' if you have GPU
    random_state=42
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

[0]	validation_0-auc:0.93952
[100]	validation_0-auc:0.95798
[200]	validation_0-auc:0.95966
[300]	validation_0-auc:0.96011
[400]	validation_0-auc:0.96026
[499]	validation_0-auc:0.96023


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'auc'


In [38]:
#predicitons
y_pred = xgb_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)
print(f"XGBoost AUC: {auc:.5f}")
amex_score = amex_metric(y_test.values, y_pred)
print(f"XGBoost AMEX Metric: {amex_score:.5f}")

XGBoost AUC: 0.96023
XGBoost AMEX Metric: 0.78865
